In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [44]:
df = pd.read_csv("./merged_bid_ask_ohlcv_data.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)


In [45]:
# Effective Bid Ask Spread
def EffectiveBidAskSpread(data, window_size=60):
    ''' Spread is to measure liquidity, larger spread, less liquidity
    '''
    result = data[['timestamp','close']].copy()
    result['Delta_P_t'] = result['close'].diff()
    result['Delta_P_shift'] = result['Delta_P_t'].shift()
    
    result['EffectiveBidAskSpread'] = (
        result['Delta_P_t'].
                   rolling(window=window_size).cov(result['Delta_P_shift']).
                   apply(lambda x: max(0,(-x))**0.5)
    )
    data['EffectiveSpread'] = result['EffectiveBidAskSpread']
    
    return data
df = EffectiveBidAskSpread(df) # add Effective Spread

In [46]:
# High Low Volatility
def HLVolatility1(data, window_size=60):
    result = pd.DataFrame()
    result['timestamp'] = data['timestamp']
    
    result['HL_Volatility'] = (np.log(data['high'])-np.log(data['low']))**2
    result['HL_Volatility'] = result['HL_Volatility'].rolling(window=window_size).mean()
    result['HL_Volatility'] = (result['HL_Volatility']/(4*np.log(2)))**0.5
    df['HLVolatility'] = result['HL_Volatility']
    
    return df
df = HLVolatility1(df)

In [47]:
# Corwin-Schultz Spread
def CSSpread(data, window_size=60):
    result = pd.DataFrame()
    result['timestamp'] = data['timestamp']
    result['gama'] = (np.log(np.maximum(data['high'], data['high'].shift(1))/np.minimum(data['high'], data['high'].shift(1))))**2
    
    result['beta'] = (np.log(data['high']/data['low']))**2
    result['beta'] = (result['beta']+result['beta'].shift(1)).rolling(window=window_size).mean()
    
    result['alpha'] = result['beta']**0.5 * (2**0.5-1) / (3-2*2**0.5) - (result['gama'] / (3-2*2**0.5))**0.5
    result['alpha'] = result['alpha'].apply(lambda x:max(x,0)) # avoid negative spread
    
    result['S'] = 2*(np.exp(result['alpha'])-1)/(np.exp(result['alpha'])+1)
    df['CS_Spread'] = result['S']
    
    return df
df = CSSpread(df)

In [48]:
import statsmodels.api as sm
# Kyle's Lambda -> 1/lambda implies liquidity
def Kyle_Lambda(data):
    df = pd.DataFrame()
    df['date'] = data['timestamp'].dt.date
    df['delta_p_t'] = data['close'] - data['close'].shift(1)
    df['delta_price'] = data['close'] - data['close'].shift(1)
    df['b_t'] = np.sign(df['delta_price'])
    df['b_t'] = df['b_t'].replace(0, method='ffill')
    df['b_t'] = df['b_t'].fillna(1)
    df['signed_volume'] = df['b_t'] * data['volume']
    kyle_lambda_list = []
    grouped = df.dropna(subset=['delta_p_t', 'signed_volume']).groupby('date')
    for _, group in grouped:
        if len(group) > 1:
            X, y = group['signed_volume'], group['delta_p_t']
            X = sm.add_constant(X)
            model = sm.OLS(y, X)
            results = model.fit()
            lambda_hat = results.params['signed_volume']
            df.loc[group.index, 'Kyle_Lambda'] = lambda_hat
        else:
            df.loc[group.index, 'Kyle_Lambda'] = np.nan
    data['Kyle_Lambda'] = df['Kyle_Lambda']
    return data
df = Kyle_Lambda(df)

/var/folders/n4/y00km0293ls6k971pcx61wnr0000gn/T/ipykernel_51944/2997099337.py:9: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df['b_t'] = df['b_t'].replace(0, method='ffill')


In [49]:
# Volume-Synchronized Probability of Informed Trading (VPIN) -> lead volatility
def VolumePIN(data, window_size=60):
    df = pd.DataFrame()
    df['Total_Bid_Size'] = data[['bid_size_1', 'bid_size_2', 'bid_size_3', 'bid_size_4', 'bid_size_5']].sum(axis=1)
    df['Total_Ask_Size'] = data[['ask_size_1', 'ask_size_2', 'ask_size_3', 'ask_size_4', 'ask_size_5']].sum(axis=1)
    df['OBI'] = (df['Total_Bid_Size'] - df['Total_Ask_Size']) / (df['Total_Bid_Size'] + df['Total_Ask_Size'])
    df['OBI'] = df['OBI'].replace([np.inf, -np.inf], np.nan).fillna(0)
    df['V_B'] = data['volume'] * (1 + df['OBI']) / 2 # estimate V^B
    df['V_S'] = data['volume'] * (1 - df['OBI']) / 2 # estimate V^S
    df['sum_V_B'] = df['V_B'].rolling(window=window_size).sum()
    df['sum_V_S'] = df['V_S'].rolling(window=window_size).sum()
    df['V'] = data['volume'].rolling(window=window_size).mean()
    df['numerator'] = (df['V_B'] - df['V_S']).abs().rolling(window=window_size).sum()
    df['denominator'] = window_size * df['V']
    df['denominator'] = df['denominator'].replace(0, np.nan)
    data['Volume_PIN'] = df['numerator'] / df['denominator']
    return data
df = VolumePIN(df)

In [50]:
# Fibonacci Retracement (maybe not so useful) -> support and resistance
def Fibo_Retrace(data, window_size=390):
    period_high = data['high'].rolling(window=window_size, min_periods=1).max()
    period_low = data['low'].rolling(window=window_size, min_periods=1).min()
    for i,level in enumerate([0.236, 0.382, 0.5, 0.618, 0.786]):
        data[f'Fib_Level_{i}'] = period_high - (period_high - period_low) * level
    return data
df = Fibo_Retrace(df)

In [51]:
df.iloc[100:110,-10:]

,EffectiveSpread,HLVolatility,CS_Spread,Kyle_Lambda,Volume_PIN,Fib_Level_0,Fib_Level_1,Fib_Level_2,Fib_Level_3,Fib_Level_4
100,0.0,0.000645,0.001901,0.000001,0.286293,178.74356,178.26322,177.875,177.48678,176.93406
101,0.0,0.000645,0.002708,0.000001,0.286614,178.74356,178.26322,177.875,177.48678,176.93406
102,0.0,0.000644,0.003664,0.000001,0.276853,178.74356,178.26322,177.875,177.48678,176.93406
103,0.0,0.000645,0.003392,0.000001,0.278046,178.74356,178.26322,177.875,177.48678,176.93406
104,0.0,0.000643,0.003663,0.000001,0.283555,178.74356,178.26322,177.875,177.48678,176.93406
105,0.0,0.000641,0.003651,0.000001,0.283426,178.74356,178.26322,177.875,177.48678,176.93406
106,0.0,0.000639,0.002406,0.000001,0.284223,178.74356,178.26322,177.875,177.48678,176.93406
107,0.0,0.000638,0.003355,0.000001,0.283809,178.74356,178.26322,177.875,177.48678,176.93406
108,0.0,0.000636,0.003072,0.000001,0.281541,178.74356,178.26322,177.875,177.48678,176.93406
109,0.0,0.000635,0.003612,0.000001,0.281605,178.74356,178.26322,177.875,177.48678,176.93406


In [55]:
# PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [57]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df.drop(columns=['timestamp','symbol']).dropna())
pca = PCA(n_components=0.95, whiten=True)
df_pca = pca.fit_transform(df_scaled)

In [61]:
df_pca = pd.DataFrame(df_pca, columns=[f'PC{i+1}' for i in range(df_pca.shape[1])])

### Reduce the sample space from 40 to 11.

In [62]:
df_pca.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11
0,-0.721715,-0.841195,-0.438455,0.349423,-0.879362,-0.859075,-0.924684,0.512443,-0.743122,-0.382969,0.091198
1,-0.754844,-0.352074,-0.132352,0.808529,-0.428902,0.618425,-0.602751,0.139250,-1.235214,-0.583261,2.011003
2,-0.737532,-0.623757,0.159126,0.696161,-0.017794,1.020565,-0.568360,0.185737,-1.265953,-0.627798,1.220056
3,-0.752353,-0.431801,-0.035609,0.105487,1.411377,2.816725,-0.726755,0.338953,-1.020472,-0.415737,-0.559806
4,-0.738441,-0.459832,0.184880,0.786086,-1.052397,-0.712263,-0.900904,0.316035,-0.984569,-0.503548,1.033149


In [60]:
df.drop(columns=['timestamp','symbol']).head()

,bid_price_1,bid_price_2,bid_price_3,bid_price_4,bid_price_5,bid_size_1,bid_size_2,bid_size_3,bid_size_4,bid_size_5,...,EffectiveSpread,HLVolatility,CS_Spread,Kyle_Lambda,Volume_PIN,Fib_Level_0,Fib_Level_1,Fib_Level_2,Fib_Level_3,Fib_Level_4
0,179.66,179.66,179.66,179.66,179.66,27600.0,30000.0,23100.0,23100.0,43100.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,179.89,179.89,179.89,179.89,179.89,21200.0,900.0,1200.0,20300.0,300.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,180.12,180.12,180.12,180.12,180.11,10000.0,30000.0,20000.0,30000.0,10000.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,179.99,179.99,179.99,179.99,179.99,17400.0,19700.0,9700.0,19700.0,9700.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,179.78,179.78,179.78,179.78,179.78,10000.0,10000.0,12100.0,10600.0,10600.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
feature_contributions = pca.components_
feature_contributions

array([[ 2.24000291e-01,  2.24000226e-01,  2.24000092e-01,
         2.23999990e-01,  2.23999881e-01, -4.07980593e-02,
        -4.26397390e-02, -4.08607162e-02, -4.09221769e-02,
        -4.04867285e-02,  2.23952846e-01,  2.23952924e-01,
         2.23953071e-01,  2.23953182e-01,  2.23953304e-01,
        -3.82808177e-02, -4.13500091e-02, -4.12060719e-02,
        -3.97872544e-02, -3.98305082e-02,  0.00000000e+00,
         0.00000000e+00,  8.06188514e-02,  2.23977738e-01,
         2.24000764e-01,  2.23952233e-01,  2.23975334e-01,
         3.54598915e-03, -6.46478452e-04,  4.75552601e-02,
         3.47905002e-02,  5.65662707e-02,  4.80928962e-02,
         1.09542779e-01,  6.00603901e-02,  2.23894647e-01,
         2.23822781e-01,  2.23709253e-01,  2.23544651e-01,
         2.23219565e-01],
       [ 3.42872408e-02,  3.42888549e-02,  3.42918893e-02,
         3.42942921e-02,  3.42970311e-02,  2.20274317e-01,
         2.40981069e-01,  2.58419811e-01,  2.58983042e-01,
         2.52017358e-01,  3.49